# In Class Assignment 2026.04.21

In [7]:
import pandas as pd
import numpy as np
import optuna
from xgboost import XGBClassifier

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.base import clone
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_recall_fscore_support, RocCurveDisplay, roc_auc_score, cohen_kappa_score, f1_score
import matplotlib.pyplot as plt

## Feature Engineering

In [9]:
df = pd.read_csv('adult.csv')

# replace ? with np.nan
df = df.replace("?", np.nan)
df.head()

# convert target variable to binary
df["income"] = df["income"].apply(lambda x: 1 if x == ">50K" else 0)

# convert gender to 0/1 (doesn't need categorical encoding since it's binary)
if "gender" in df.columns:
    df["gender"] = df["gender"].apply(lambda x: 1 if x == "Male" else 0)  

df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,0,0,0,30,United-States,0


In [10]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

age: 74 unique values
workclass: 8 unique values
fnlwgt: 28523 unique values
education: 16 unique values
educational-num: 16 unique values
marital-status: 7 unique values
occupation: 14 unique values
relationship: 6 unique values
race: 5 unique values
gender: 2 unique values
capital-gain: 123 unique values
capital-loss: 99 unique values
hours-per-week: 96 unique values
native-country: 41 unique values
income: 2 unique values


### Filling NaN's (for KNN model)

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   age              48842 non-null  int64
 1   workclass        46043 non-null  str  
 2   fnlwgt           48842 non-null  int64
 3   education        48842 non-null  str  
 4   educational-num  48842 non-null  int64
 5   marital-status   48842 non-null  str  
 6   occupation       46033 non-null  str  
 7   relationship     48842 non-null  str  
 8   race             48842 non-null  str  
 9   gender           48842 non-null  int64
 10  capital-gain     48842 non-null  int64
 11  capital-loss     48842 non-null  int64
 12  hours-per-week   48842 non-null  int64
 13  native-country   47985 non-null  str  
 14  income           48842 non-null  int64
dtypes: int64(8), str(7)
memory usage: 8.9 MB


I notice that there are an near equal number of null occupation values as there are for workforce values, indicating that some of the people are unemployed.  Let's check:

In [12]:
df[df["occupation"].isna()].info()

<class 'pandas.DataFrame'>
Index: 2809 entries, 4 to 48823
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   age              2809 non-null   int64
 1   workclass        10 non-null     str  
 2   fnlwgt           2809 non-null   int64
 3   education        2809 non-null   str  
 4   educational-num  2809 non-null   int64
 5   marital-status   2809 non-null   str  
 6   occupation       0 non-null      str  
 7   relationship     2809 non-null   str  
 8   race             2809 non-null   str  
 9   gender           2809 non-null   int64
 10  capital-gain     2809 non-null   int64
 11  capital-loss     2809 non-null   int64
 12  hours-per-week   2809 non-null   int64
 13  native-country   2763 non-null   str  
 14  income           2809 non-null   int64
dtypes: int64(8), str(7)
memory usage: 488.8 KB


Indeed, anyone with a null occupation has a null workclass.  What about the 10 people who have a non-null occupation but a null workclass?

In [13]:
df[(df["workclass"].notna()) & (df["occupation"].isna())]

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
8785,17,Never-worked,131593,11th,7,Never-married,NaN,Own-child,Black,0,0,0,20,United-States,0
11607,20,Never-worked,273905,HS-grad,9,Married-spouse-absent,NaN,Other-relative,White,1,0,0,35,United-States,0
13898,18,Never-worked,162908,11th,7,Never-married,NaN,Own-child,White,1,0,0,35,United-States,0
21642,18,Never-worked,206359,10th,6,Never-married,NaN,Own-child,White,1,0,0,40,United-States,0
27126,23,Never-worked,188535,7th-8th,4,Divorced,NaN,Not-in-family,White,1,0,0,35,United-States,0
31053,17,Never-worked,237272,10th,6,Never-married,NaN,Own-child,White,1,0,0,30,United-States,0
36618,18,Never-worked,157131,11th,7,Never-married,NaN,Own-child,White,0,0,0,10,United-States,0
39513,20,Never-worked,462294,Some-college,10,Never-married,NaN,Own-child,Black,1,0,0,40,United-States,0
48585,30,Never-worked,176673,HS-grad,9,Married-civ-spouse,NaN,Wife,Black,0,0,0,40,United-States,0
48595,18,Never-worked,153663,Some-college,10,Never-married,NaN,Own-child,White,1,0,0,4,United-States,0


All of these people have never worked, which I will group with unemployed (additionally, there is no need to create a separate class for 10 people, especially for classification problems).  I should remove NA values for the KNNClassifier.

In [14]:
df["workclass"] = df["workclass"].fillna("Unemployed")
df["occupation"] = df["occupation"].fillna("Unemployed")
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1
4,18,Unemployed,103497,Some-college,10,Never-married,Unemployed,Own-child,White,0,0,0,30,United-States,0


Now let's inspect `native-country`:

In [15]:
df[df["native-country"].isna()]

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
19,40,Private,85019,Doctorate,16,Married-civ-spouse,Prof-specialty,Husband,Asian-Pac-Islander,1,0,0,45,NaN,1
65,41,Private,109912,Bachelors,13,Never-married,Other-service,Not-in-family,White,0,0,0,40,NaN,0
83,44,Self-emp-inc,223881,HS-grad,9,Married-civ-spouse,Craft-repair,Husband,White,1,99999,0,50,NaN,1
188,34,State-gov,513100,Bachelors,13,Married-spouse-absent,Farming-fishing,Not-in-family,Black,1,0,0,40,NaN,0
253,42,Federal-gov,177937,Bachelors,13,Never-married,Prof-specialty,Not-in-family,White,1,0,0,40,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48730,44,Self-emp-inc,71556,Masters,14,Married-civ-spouse,Sales,Husband,White,1,0,0,50,NaN,1
48750,58,Self-emp-inc,181974,Doctorate,16,Never-married,Prof-specialty,Not-in-family,White,0,0,0,99,NaN,0
48773,42,Self-emp-not-inc,217597,HS-grad,9,Divorced,Sales,Own-child,White,1,0,0,50,NaN,0
48791,39,Private,107302,HS-grad,9,Married-civ-spouse,Prof-specialty,Husband,White,1,0,0,45,NaN,1


There is a variety, so I will fill the NaN's with "Unknown":

In [16]:
df["native-country"] = df["native-country"].fillna("Unknown")

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   age              48842 non-null  int64
 1   workclass        48842 non-null  str  
 2   fnlwgt           48842 non-null  int64
 3   education        48842 non-null  str  
 4   educational-num  48842 non-null  int64
 5   marital-status   48842 non-null  str  
 6   occupation       48842 non-null  str  
 7   relationship     48842 non-null  str  
 8   race             48842 non-null  str  
 9   gender           48842 non-null  int64
 10  capital-gain     48842 non-null  int64
 11  capital-loss     48842 non-null  int64
 12  hours-per-week   48842 non-null  int64
 13  native-country   48842 non-null  str  
 14  income           48842 non-null  int64
dtypes: int64(8), str(7)
memory usage: 8.9 MB


Now there are no null values! Let's feature engineer a logistic regression as an input, so that not only will we stack our XGBoost and KNN models, we will also sequentially use logistic regression beforehand.

### Logistic regression feature

In [18]:
X = df.drop("income", axis=1)
y = df["income"]
# numeric columns are dtype number except for gender
numeric_cols = ['age', 'fnlwgt', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_cols = [col for col in X.columns if col not in numeric_cols and col != "gender"]
print(categorical_cols)
logit_ct = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
], remainder="passthrough")
logit_pipeline = Pipeline([
    ("preprocessor", logit_ct),
    ("logit", LogisticRegression(random_state=42))
])
logit_param_grid = {
    "logit__C": np.logspace(-4, 4, 9),
    "logit__penalty": ["l1", "l2"],
    "logit__solver": ["liblinear"]
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logit_grid = GridSearchCV(logit_pipeline, logit_param_grid, cv=skf, n_jobs=-1, scoring="f1_macro")

['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country']


In [19]:
logit_feature = logit_grid.fit(X, y)

In [20]:
logit_feature.best_score_

np.float64(0.7832202410450906)

In [21]:
df['logit_pred'] = logit_feature.predict_proba(X)[:, 1]

In [22]:
df.head()

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income,logit_pred
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,1,0,0,40,United-States,0,0.002262
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,1,0,0,50,United-States,0,0.125018
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,1,0,0,40,United-States,1,0.394462
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,1,7688,0,40,United-States,1,0.761728
4,18,Unemployed,103497,Some-college,10,Never-married,Unemployed,Own-child,White,0,0,0,30,United-States,0,0.001617


We have successfully imputed the null values and created a feature using logistic regression that will be an input feature for out XGBoost and KNN models.

(Yes, it is kind of cheating to use a logistic regression when there is no external validation data, because then the XGBoost and KNN models have a new input feature that has high correlation with the output variable, but I'm sure it's not an issue for the purposes of this in-class assignment.)

## Model Building

In [23]:
X = df.drop("income", axis=1)
y = df["income"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### XGBoost

In [24]:
# convert string columns to categorical for XGBoost
for col in X_train.columns:
    if X_train[col].dtype == "str":
        X_train[col] = X_train[col].astype("category")
        X_test[col] = X_test[col].astype("category")

In [25]:
xgb_model = XGBClassifier(random_state=42, enable_categorical=True, use_label_encoder=False, eval_metric='logloss')
xgb_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}
xgb_grid = GridSearchCV(xgb_model, xgb_param_grid, cv=skf, n_jobs=-1, scoring="f1_macro")

In [26]:
xgb_grid.fit(X_train, y_train)

c:\Users\jfigg\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [23:22:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,estimator,"XGBClassifier...ree=None, ...)"
,param_grid,"{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.01, 0.1, ...], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200, ...], ...}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,objective,'binary:logistic'


In [27]:
xgb_grid.best_score_

np.float64(0.8165489302208065)

In [ ]:
f1_score(y_test, xgb_grid.predict(X_test))

### KNN

In [28]:
knn_ct = ColumnTransformer([
    ("standardizer", StandardScaler(), numeric_cols),
    ("ohe", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
], remainder="passthrough")
knn_pipeline = Pipeline([
    ("preprocessor", knn_ct),
    ("knn", KNeighborsClassifier())
])
knn_param_grid = {
    "knn__n_neighbors": np.arange(1, 10),
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan"]
}
knn_grid = GridSearchCV(knn_pipeline, knn_param_grid, cv=skf, scoring="f1_macro")

In [29]:
knn_grid.fit(X_train, y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'knn__metric': ['euclidean', 'manhattan'], 'knn__n_neighbors': array([1, 2, ..., 6, 7, 8, 9]), 'knn__weights': ['uniform', 'distance']}"
,scoring,'f1_macro'
,n_jobs,None
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('standardizer', ...), ('ohe', ...)]"


In [30]:
knn_grid.best_score_

np.float64(0.7707448321152246)

## Combining Weights with Optuna

In [31]:
def objective(trial):
    knn_weight = trial.suggest_float("knn_weight", 0.0, 1.0)
    xgb_weight = 1.0 - knn_weight

    knn_pred = knn_grid.predict_proba(X_test)[:, 1]
    xgb_pred = xgb_grid.predict_proba(X_test)[:, 1]

    ensemble_pred = knn_weight * knn_pred + xgb_weight * xgb_pred
    ensemble_pred_binary = (ensemble_pred >= 0.5).astype(int)

    return f1_score(y_test, ensemble_pred_binary, average="macro")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

[I 2026-04-21 23:42:48,518] A new study created in memory with name: no-name-81fb76e9-cb99-4524-afed-c60cf6f4e972
[I 2026-04-21 23:43:04,744] Trial 0 finished with value: 0.8170301215428533 and parameters: {'knn_weight': 0.2929855725097782}. Best is trial 0 with value: 0.8170301215428533.
[I 2026-04-21 23:43:20,697] Trial 1 finished with value: 0.8164858683343656 and parameters: {'knn_weight': 0.026982319301342206}. Best is trial 0 with value: 0.8170301215428533.
[I 2026-04-21 23:43:36,475] Trial 2 finished with value: 0.7773779922721265 and parameters: {'knn_weight': 0.994660775774186}. Best is trial 0 with value: 0.8170301215428533.
[I 2026-04-21 23:43:52,300] Trial 3 finished with value: 0.8172099378504395 and parameters: {'knn_weight': 0.17505133952376817}. Best is trial 3 with value: 0.8172099378504395.
[I 2026-04-21 23:44:08,201] Trial 4 finished with value: 0.8129902924577708 and parameters: {'knn_weight': 0.5429399531087613}. Best is trial 3 with value: 0.8172099378504395.
[I 2

In [32]:
study.best_value

0.818525013126892

In [40]:
# feature importance for XGBoost with feature names
xgb_importance = xgb_grid.best_estimator_.feature_importances_
feature_names = xgb_grid.best_estimator_.get_booster().feature_names
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": xgb_importance
}).sort_values(by="importance", ascending=False)
importance_df

,feature,importance
14,logit_pred,0.602729
10,capital-gain,0.084334
11,capital-loss,0.065913
0,age,0.029150
5,marital-status,0.026599
6,occupation,0.023348
7,relationship,0.022799
12,hours-per-week,0.021857
1,workclass,0.020715
3,education,0.020337


As expected, the logistic regression is the most important feature.

XGBoost did perform a bit better than KNN, with an `f1_macro` score of around .817 compared to the KNN score of around .77.

As we see, the combination of KNN and XGBoost created slight improvement compared to using only 1 of the models (note that I used `f1_macro` as the scoring metric due to slightly unbalanced data)